# Drift Exploration

A visual walkthrough of the same comparison `src/main.py` performs: profile the
training baseline, compare an incoming batch against it, and see which drift
signal fires for which feature.

Run this notebook from anywhere - paths are resolved relative to the repository
root, not a hardcoded machine path.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "src"))
DATA = REPO_ROOT / "data"

import pandas as pd

from drift_detector import detect_drift
from monitor import monitor_system
from stats_profile import compute_profile

print(f"Repository root: {REPO_ROOT}")

## 1. Load the datasets

`train.csv` is the training-time baseline. `new_data.csv` is an incoming batch
drawn from a shifted population; `stable_data.csv` is drawn from the same
population as the baseline and acts as the control.


In [ ]:
baseline = pd.read_csv(DATA / "train.csv")
drifted = pd.read_csv(DATA / "new_data.csv")
stable = pd.read_csv(DATA / "stable_data.csv")

print(f"baseline: {baseline.shape}  drifted: {drifted.shape}  stable: {stable.shape}")
baseline.head()

## 2. Profile the baseline

This is the snapshot you would persist alongside a trained model.


In [ ]:
pd.DataFrame(compute_profile(baseline)).T

## 3. Compare the two batches side by side


In [ ]:
pd.DataFrame({
    "baseline": baseline.mean(),
    "stable": stable.mean(),
    "drifted": drifted.mean(),
}).round(2)

## 4. Run drift detection

Note `credit_score` in the drifted batch: its mean moves by less than the 10%
threshold, so mean drift alone would call it stable. The KS test still rejects
the null hypothesis - which is exactly why both signals exist.


In [ ]:
drift_report = detect_drift(baseline, drifted)
pd.DataFrame([r.to_dict() for r in drift_report.values()]).set_index("feature").round(4)

In [ ]:
stable_report = detect_drift(baseline, stable)
pd.DataFrame([r.to_dict() for r in stable_report.values()]).set_index("feature").round(4)

## 5. System-level verdict


In [ ]:
for label, report in [("drifted batch", drift_report), ("stable batch", stable_report)]:
    status = monitor_system(report)
    print(f"{label:>14}: {status.status:<14} "
          f"({len(status.drifted_features)}/{status.total_features} features, "
          f"ratio {status.drift_ratio:.0%})")

## 6. Visualise the shift

Overlaid histograms make the distribution change obvious for every feature.


In [ ]:
import matplotlib.pyplot as plt

features = list(drift_report)
fig, axes = plt.subplots(1, len(features), figsize=(5 * len(features), 4))

for ax, feature in zip(axes, features):
    ax.hist(baseline[feature], bins=40, alpha=0.6, label="baseline")
    ax.hist(drifted[feature], bins=40, alpha=0.6, label="incoming")
    ax.set_title(f"{feature} (p = {drift_report[feature].p_value:.3g})")
    ax.legend()

plt.tight_layout()
plt.show()